In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from collections import Counter
import os
from tqdm import tqdm

In [16]:
full_df = pd.read_excel("/content/drive/MyDrive/Colab_Notebooks/NLP/DATASET/Training.xlsx")

def map_emotion(label):
    if label == '기쁨':
        return 1
    elif label in ['슬픔', '상처']:
        return 0
    elif label == '분노':
        return 2
    elif label in ['당황', '불안']:
        return 3
    else:
        return -1

In [17]:
rows = []

for idx, row in full_df.iterrows():
    if pd.notnull(row['사람문장1']) and pd.notnull(row['시스템문장1']):
        rows.append({
            'input_text': row['사람문장1'],
            'target_text': row['시스템문장1'],
            'emotion_label': map_emotion(row['감정_대분류'])
        })
    if pd.notnull(row['사람문장2']) and pd.notnull(row['시스템문장2']):
        rows.append({
            'input_text': row['사람문장2'],
            'target_text': row['시스템문장2'],
            'emotion_label': map_emotion(row['감정_대분류'])
        })
    if '시스템문장3' in row and pd.notnull(row['사람문장3']) and pd.notnull(row['시스템문장3']):
        rows.append({
            'input_text': row['사람문장3'],
            'target_text': row['시스템문장3'],
            'emotion_label': map_emotion(row['감정_대분류'])
        })

In [18]:
gen_df = pd.DataFrame(rows)
gen_df = gen_df[gen_df['emotion_label'] != -1]  # 유효 레이블만 사용

print(f"생성된 학습 데이터 개수: {len(gen_df)}")
gen_df.head()

생성된 학습 데이터 개수: 145954


,input_text,target_text,emotion_label
0,일은 왜 해도 해도 끝이 없을까? 화가 난다.,많이 힘드시겠어요. 주위에 의논할 상대가 있나요?,2
1,그냥 내가 해결하는 게 나아. 남들한테 부담 주고 싶지도 않고.,혼자 해결하기로 했군요. 혼자서 해결하기 힘들면 주위에 의논할 사람을 찾아보세요.,2
2,이번 달에 또 급여가 깎였어! 물가는 오르는데 월급만 자꾸 깎이니까 너무 화가 나.,급여가 줄어 속상하시겠어요. 월급이 줄어든 것을 어떻게 보완하실 건가요?,2
3,최대한 지출을 억제해야겠어. 월급이 줄어들었으니 고정지출을 줄일 수밖에 없을 것 같아.,월급이 줄어든 만큼 소비를 줄일 계획이군요.,2
4,회사에 신입이 들어왔는데 말투가 거슬려. 그런 애를 매일 봐야 한다고 생각하니까 스...,회사 동료 때문에 스트레스를 많이 받는 것 같아요. 문제 해결을 위해 어떤 노력을 ...,2


In [19]:
counter = Counter()
for text in gen_df['input_text']:
    counter.update(text.split())
for text in gen_df['target_text']:
    counter.update(text.split())

vocab = {"<pad>": 0, "<unk>": 1, "<sos>": 2, "<eos>": 3}
for idx, (word, _) in enumerate(counter.items(), start=4):
    vocab[word] = idx
vocab_size = len(vocab)
print(f"vocab size: {vocab_size}")

import pickle

with open("/content/drive/MyDrive/Colab_Notebooks/NLP/sen_vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

print("vocab.pkl 저장 완료!")

vocab size: 153591
vocab.pkl 저장 완료!


In [23]:
def encode_text(text, max_len=120, add_tokens=False):
    tokens = text.split()
    if add_tokens:
        tokens = ['<sos>'] + tokens + ['<eos>']
    token_ids = [vocab.get(token, vocab["<unk>"]) for token in tokens]
    return token_ids[:max_len] + [vocab["<pad>"]] * (max_len - len(token_ids))

# 인코딩 적용
X = [encode_text(text) for text in gen_df['input_text']]
Y = [encode_text(text, add_tokens=True) for text in gen_df['target_text']]
emotion_labels = gen_df['emotion_label'].tolist()

In [24]:
class GenDataset(Dataset):
    def __init__(self, X, Y, emotions):
        self.X = torch.tensor(np.array(X), dtype=torch.long)
        self.Y = torch.tensor(np.array(Y), dtype=torch.long)
        self.emotions = torch.tensor(np.array(emotions), dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx], self.emotions[idx]

In [25]:
dataset = GenDataset(X, Y, emotion_labels)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [26]:
import sys
sys.path.append('/content/drive/MyDrive/Colab_Notebooks/NLP')

In [27]:
from sen_gen_model import EmotionConditionedSeq2Seq

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 모델 선언
gen_model = EmotionConditionedSeq2Seq(
    vocab_size=vocab_size,
    embed_dim=200,
    hidden_dim=256,
    emotion_dim=32,
    sos_id=vocab.get('<sos>', 2),
    eos_id=vocab.get('<eos>', 3)
).to(device)

Using device: cuda


In [28]:
# Optimizer, Loss
optimizer = optim.Adam(gen_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

In [29]:
# 학습 루프 + early stopping + best model 저장

num_epochs = 20  # ← 15~20 추천 (시연용)
patience = 3
no_improve_epochs = 0
best_loss = float('inf')

for epoch in range(num_epochs):
    gen_model.train()
    total_loss = 0

    loop = tqdm(loader, desc=f"[Train] Epoch {epoch+1}")
    for X_batch, Y_batch, emo_batch in loop:
        X_batch, Y_batch, emo_batch = X_batch.to(device), Y_batch.to(device), emo_batch.to(device)

        logits = gen_model(X_batch, emo_batch, Y_batch)
        loss = criterion(logits.view(-1, logits.shape[-1]), Y_batch.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")

    # Early stopping + best model 저장
    if avg_loss < best_loss:
        best_loss = avg_loss
        save_path = "/content/drive/MyDrive/Colab_Notebooks/NLP/gen_model.pt"
        torch.save(gen_model.state_dict(), save_path)
        print("✅ Best model saved!")
        no_improve_epochs = 0
    else:
        no_improve_epochs += 1
        print(f"⚠️ No improvement for {no_improve_epochs} epoch(s)")

    if no_improve_epochs >= patience:
        print("🛑 Early stopping triggered.")
        break


[Train] Epoch 1: 100%|██████████| 4562/4562 [06:36<00:00, 11.50it/s]


Epoch 1/20 - Loss: 0.4938
✅ Best model saved!


[Train] Epoch 2: 100%|██████████| 4562/4562 [06:36<00:00, 11.51it/s]


Epoch 2/20 - Loss: 0.1764
✅ Best model saved!


[Train] Epoch 3: 100%|██████████| 4562/4562 [06:36<00:00, 11.50it/s]


...

Epoch 18/20 - Loss: 0.0000
✅ Best model saved!


[Train] Epoch 19: 100%|██████████| 4562/4562 [06:37<00:00, 11.48it/s]


Epoch 19/20 - Loss: 0.0000
✅ Best model saved!


[Train] Epoch 20: 100%|██████████| 4562/4562 [06:37<00:00, 11.48it/s]


Epoch 20/20 - Loss: 0.0000
✅ Best model saved!


In [30]:
# 저장
save_path = "/content/drive/MyDrive/Colab_Notebooks/NLP/gen_model_2.pt"
torch.save(gen_model.state_dict(), save_path)
print("gen_model.pt 저장 완료!")

gen_model.pt 저장 완료!
